<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Supplementary code for the <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> book by <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Code repository: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>


# Ek D: Daha büyük LLM'ler kullanmak

Bu not defterinde kullanılan paketler:

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- Ana bölümler Qwen3 0.6B temel modelini kullanır; çünkü Qwen3 ailesindeki en küçük model odur
ve dolayısıyla tüketici donanımında çalıştırması en kolay olandır
- Ancak Ek C'deki aynı `Qwen3Model` uygulaması, aynı sıfırdan PyTorch model koduyla daha büyük yoğun Qwen3 kontrol noktalarını yüklemek için de kullanılabilir

&nbsp;
## D.1 Daha büyük yoğun Qwen3 yapılandırmaları

Depo, `reasoning_from_scratch.appendix_c` ([reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)) içinde (0.6B modelinin ötesinde) daha büyük birkaç yoğun Qwen3 modeli için
yapılandırma sözlükleri barındırır:

| Model boyutu | Yapılandırma sözlüğü |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- Yukarıdaki şekilde belirtildiği gibi, bunlar tek GPU'larda çalışabilen "yoğun" Qwen3 çeşitleridir
- Qwen3'ün "seyrek" Uzmanlar Karışımı (Mixture-of-Experts) çeşitleri de vardır, ancak bu kitabın koduyla desteklenmezler; yine de sıfırdan bir uygulamayla ilgileniyorsanız burada bulabilirsiniz: https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- Bunların tümü, Ek C'deki 0.6B modeliyle aynı genel mimari örüntüsünü kullanır
- Değişen şeyler gömme boyutu, katman sayısı, dikkat başlığı sayısı ve
ileri besleme gizli boyutudur

- Kabaca bir alt sınır olarak, ağırlıkları bfloat16 biçiminde saklamak parametre başına yaklaşık 2 bayt gerektirir
- Bu, yalnızca kontrol noktası ağırlıklarının şu mertebede olduğu anlamına gelir:

| Model boyutu | bfloat16'da kabaca ağırlık belleği |
| --- | --- |
| 1.7B | yaklaşık 3.4 GB |
| 4B | yaklaşık 8 GB |
| 8B | yaklaşık 16 GB |
| 14B | yaklaşık 28 GB |
| 32B | yaklaşık 64 GB |


- Pratikte gerçek çalışma zamanı bellek kullanımı daha yüksektir; çünkü etkinleştirmeler, geçici tamponlar ve çoğu zaman KV önbelleği için de
belleğe ihtiyacımız olur

&nbsp;
## D.2 Daha büyük kontrol noktalarını indirmeye genel bakış

- Ana bölümlerde kullanılan 0.6B kontrol noktalarının aksine, daha büyük resmî Qwen3 modelleri
tipik olarak `safetensors` dosyaları hâlinde, bazen birden çok parçaya bölünmüş olarak dağıtılır
- Bunları yüklemeye yarayan `download_from_huggingface_from_snapshots` yardımcı fonksiyonu bazı ek paketler gerektirir:

```bash
!uv add huggingface_hub safetensors
```

or

```bash
!pip install huggingface_hub safetensors
```

&nbsp;
## D.3 Daha büyük bir temel model yüklemek

- Ağırlıkları indir:

In [2]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

Using Apple Silicon GPU (MPS)


/Users/sebastian/Developer/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 2616.79it/s]


- Modeli başlat:

In [3]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

Model uses weight tying.


Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

- Tokenizer'ı yükle:

In [4]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# Özgün temel tokenizer'ın adının "tokenizer.json" olduğunu unutmayın
# Akıl yürütme tokenizer'ından (sonraki bölüm) ayırmak için yeniden adlandırıyoruz
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- Modeli kullan:

In [5]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

 Large language models are artificial intelligence systems that use deep learning techniques to understand and generate human-like text. They are trained on vast amounts of data and can perform a wide range of natural language processing tasks, such as translation, summarization, and question answering.

&nbsp;
## D.4 Daha büyük bir akıl yürütme çeşidini yüklemek

- Aynı fikir daha büyük akıl yürütme tarzı Qwen3 modelleri için de geçerlidir
- Belirli bir model boyutu için mimari aynı kalır; yalnızca kontrol noktası ve tokenizer ayarları değişir

Örneğin, 4B temel çeşidi yerine 4B akıl yürütme çeşidini yüklemek için şunları yapardık:

- depo kimliğini `Qwen/Qwen3-4B-Base` yerine `Qwen/Qwen3-4B` yapmak;
- `tokenizer.json` dosyasını `tokenizer-reasoning.json` olarak kopyalamak;
- tokenizer'ı şöyle başlatmak:

```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- Modeli yükleme ve kullanma kodunun geri kalanı aynı kalır

&nbsp;
## D.5 Pratik öneriler

- Bu bölümde kod yok